# 04 — Statistical Testing (Step 4.15)

Tests the three findings I was least confident about, from `reports/report_analysis.md`:

1. **G3** — the toss-decision win-rate gap
2. **I1 (reframed)** — whether venue is secretly driving the H2 chase-win-rate trend
3. **Specialism** — whether AB de Villiers' death-overs strike rate is a real effect or a small-sample fluke

For each: state the finding → write a sharp statistical question → write the null hypothesis →
choose the test → run it → record p-value, effect size, n → write a plain-English verdict. The
null hypothesis is written before the test is run, not after seeing the result.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, mannwhitneyu

con = sqlite3.connect("../ipl.db")
pd.set_option("display.width", 120)

## Test 1 — G3: Does the toss decision actually matter?

**Move 1 — Finding:** Toss winners who chose to field won 54.6% of the time vs. 45.8% for those
who chose to bat.

**Move 2 — Statistical question:** Is there a significant association between the toss winner's
decision (bat vs. field) and whether the toss winner goes on to win the match?

**Move 3 — Null hypothesis (H0):** The toss winner's win/loss outcome is independent of their toss
decision; there is no association between the two.

**Move 4 — Test chosen:** Both variables are categorical → **chi-square test of independence**,
with the phi coefficient as the effect size (appropriate for a 2×2 table).

In [2]:
m = pd.read_sql("SELECT toss_winner, toss_decision, match_winner, result FROM matches WHERE result='win'", con)
m["toss_winner_won"] = np.where(m.toss_winner == m.match_winner, "Won", "Lost")

table1 = pd.crosstab(m["toss_decision"], m["toss_winner_won"])
table1

toss_winner_won,Lost,Won
toss_decision,,
bat,218,184
field,356,429


In [3]:
chi2_1, p1, dof1, expected1 = chi2_contingency(table1)
n1 = table1.values.sum()
phi1 = np.sqrt(chi2_1 / n1)

print(f"chi-square = {chi2_1:.3f}, dof = {dof1}")
print(f"p-value    = {p1:.6f}")
print(f"n          = {n1}  (402 chose bat, 785 chose field)")
print(f"phi        = {phi1:.4f}  (below Cohen's 'small' threshold of 0.10)")

chi-square = 8.040, dof = 1
p-value    = 0.004575
n          = 1187  (402 chose bat, 785 chose field)
phi        = 0.0823  (below Cohen's 'small' threshold of 0.10)


**Move 6 — Verdict:** The test provides evidence of a real association between the toss decision
and the outcome (p = 0.0046), but the effect size (φ = 0.082) is very small — the toss decision
moves the odds of winning only slightly. Statistically real, but should not be oversold as a strong
lever. **Status: survived, small effect.**

## Test 2 — I1 (reframed): Is venue secretly driving the H2 chase-win-rate trend?

**Move 1 — Finding:** H2 pools every venue together when showing chase win rate falling as the
target rises. The underlying I1 doubt — that dirty venue names hide real per-venue differences —
implies a sharper worry: could a handful of chase-friendly grounds be driving the whole pooled H2
trend, rather than target size itself?

**Move 2 — Statistical question:** Is there a significant association between venue and whether
the chasing team won the match?

**Move 3 — Null hypothesis (H0):** The chase win/loss outcome is independent of venue; chase win
rate does not differ meaningfully across grounds.

**Move 4 — Test chosen:** Two categorical variables → **chi-square test of independence**,
restricted to the cleaned venues with at least 30 matches (so no cell rests on a handful of games),
with Cramér's V as the effect size.

In [4]:
inn1 = pd.read_sql("SELECT match_id, SUM(total_runs) AS inn1_runs FROM deliveries WHERE innings=1 GROUP BY match_id", con)
mm = pd.read_sql("SELECT match_id, venue, win_by_wickets, result FROM matches", con)

def clean_venue(x):
    base = x.split(",")[0]
    base = base.replace(".", " ")
    return " ".join(base.split())

mm["venue_clean"] = mm.venue.apply(clean_venue)
h = mm.merge(inn1, on="match_id").query("result == 'win'").copy()
h["chase_won"] = np.where(h.win_by_wickets > 0, "Chase won", "Defended")

venue_counts = h.venue_clean.value_counts()
top_venues = venue_counts[venue_counts >= 30].index.tolist()
print(f"venues with n>=30 matches: {len(top_venues)}")

h2f = h[h.venue_clean.isin(top_venues)]
table2 = pd.crosstab(h2f["venue_clean"], h2f["chase_won"])
table2

venues with n>=30 matches: 13


chase_won,Chase won,Defended
venue_clean,,
Arun Jaitley Stadium,19,20
Dr DY Patil Sports Academy,20,17
Dubai International Cricket Stadium,22,21
Eden Gardens,57,45
Feroz Shah Kotla,31,28
M Chinnaswamy Stadium,56,44
MA Chidambaram Stadium,42,51
Maharashtra Cricket Association Stadium,17,18
Narendra Modi Stadium,19,18


In [5]:
chi2_2, p2, dof2, expected2 = chi2_contingency(table2)
n2 = table2.values.sum()
k2 = min(table2.shape)
cramers_v2 = np.sqrt((chi2_2 / n2) / (k2 - 1))

print(f"chi-square = {chi2_2:.3f}, dof = {dof2}")
print(f"p-value    = {p2:.4f}")
print(f"n          = {n2}  across {len(top_venues)} venues")
print(f"Cramer's V = {cramers_v2:.4f}  (negligible)")

chi-square = 7.866, dof = 12
p-value    = 0.7955
n          = 860  across 13 venues
Cramer's V = 0.0956  (negligible)


**Move 6 — Verdict:** The test did not provide enough evidence of a difference in chase win rate
across venues (p = 0.7955), and the observed effect was negligible (Cramér's V = 0.096) based on
n = 860. This is a useful negative result: the H2 "chasing gets harder as the target rises" finding
is **not** simply an artifact of a few chase-friendly grounds dominating the pooled sample — venue
does not appear to be a meaningful confounder. It does not resolve the separate, narrower doubt
about whether a couple of venue names still need manual merging (e.g. Punjab Cricket Association
Stadium naming) — that stays a data-cleaning question, not a statistical one.
**Status: did not survive** (as a confounder claim) — good news for H2's validity.

## Test 3 — Specialism: Is AB de Villiers' death-overs strike rate a real effect?

**Move 1 — Finding:** Among batters with ≥60 death-overs balls faced, AB de Villiers ranked 2nd by
strike rate (222.9) — but on a genuinely large sample (838 balls), unlike most names near the top
of that list, which qualify on barely more than the 60-ball minimum.

**Move 2 — Statistical question:** Is the runs-per-ball AB de Villiers scores in the death overs
significantly different from the runs-per-ball scored by the rest of the qualifying pool?

**Move 3 — Null hypothesis (H0):** The distribution of runs scored per ball in the death overs is
the same for AB de Villiers as for the rest of the qualifying batters — no difference in central
tendency.

**Move 4 — Test chosen:** Per-ball runs are a heavily skewed, discrete outcome (mostly 0s and 1s,
occasional 4s and 6s) — not normally distributed — so a **Mann–Whitney U test** (non-parametric) is
the right choice over a t-test. Effect size: rank-biserial correlation.

In [6]:
dth = pd.read_sql("SELECT batter, batter_runs, is_wide_ball FROM deliveries WHERE over_number >= 15", con)
legal = dth[dth.is_wide_ball == 0]

abd = legal[legal.batter == "AB de Villiers"].batter_runs
qualifying = legal.groupby("batter").filter(lambda g: len(g) >= 60)
rest = qualifying[qualifying.batter != "AB de Villiers"].batter_runs

print(f"AB de Villiers: n = {len(abd)}, mean runs/ball = {abd.mean():.3f}")
print(f"Rest of pool:   n = {len(rest)}, mean runs/ball = {rest.mean():.3f}")

AB de Villiers: n = 838, mean runs/ball = 2.229
Rest of pool:   n = 53877, mean runs/ball = 1.614


In [7]:
U, p3 = mannwhitneyu(abd, rest, alternative="two-sided")
n1_, n2_ = len(abd), len(rest)
rank_biserial = 1 - (2 * U) / (n1_ * n2_)

print(f"U          = {U:.1f}")
print(f"p-value    = {p3:.3e}")
print(f"n1 (ABD)   = {n1_}")
print(f"n2 (rest)  = {n2_}")
print(f"rank-biserial r = {rank_biserial:.4f}")

U          = 26414756.5
p-value    = 6.691e-19
n1 (ABD)   = 838
n2 (rest)  = 53877
rank-biserial r = -0.1701


**Move 6 — Verdict:** The test provides strong evidence that AB de Villiers' death-overs scoring
rate differs from the rest of the qualifying pool (p ≈ 6.69 × 10⁻¹⁹), with a small-to-moderate
effect size (rank-biserial r ≈ 0.17) based on n₁ = 838 and n₂ = 53,877. The extremely small p-value
is a function of the very large sample, not proof of a huge effect — the effect size is what
actually tells us how much better he was: 2.23 runs/ball vs. 1.61 for the rest, about 38% higher.
This addresses the original doubt directly: for AB de Villiers specifically, the finding survives
testing and is not a small-sample fluke. The same cannot automatically be claimed for the other
smaller-sample names on that list (e.g. R Shepherd, n = 75), which weren't individually retested.

**Alternative explanation (required — this comparison sits next to a correlation-style claim about
skill):** AB de Villiers played the bulk of his IPL career for Royal Challengers Bangalore, whose
home ground (M Chinnaswamy Stadium) is one of the highest-scoring venues in the dataset. Some of
this effect could be venue-assisted scoring rather than pure individual skill — Test 2 above shows
venue doesn't move *chase outcomes* much, but that doesn't rule out venue moving raw scoring rate,
which is a different question not tested here.

**Status: survived — real, moderate effect, specifically for AB de Villiers.**

## Summary

| Test | Question | p-value | Effect size | n | Verdict |
|---|---|---|---|---|---|
| G3 | Toss decision vs. toss-winner outcome | 0.0046 | φ = 0.082 (small) | 1,187 | Survived — real but small effect |
| I1 (reframed) | Venue vs. chase outcome | 0.7955 | Cramér's V = 0.096 (negligible) | 860 | Did not survive — good news for H2 |
| Specialism | AB de Villiers vs. rest of qualifying pool | 6.69e-19 | rank-biserial r ≈ 0.17 | 838 vs 53,877 | Survived — real, moderate effect |

Of the three findings tested: two survived (G3, Specialism) and one did not (I1 as a confounder
claim). See `reports/statistical_report.md` for the full write-up and `reports/report_analysis.md`
for each original finding's updated status.